# Evaluation: Base vs Fine-Tuned Model

This notebook is used to manually review base vs fine-tuned model responses on the test set.

**Workflow:**
1. On the GPU machine, run `python -m src.evaluation` to generate `output/results/evaluation_results.json`.
2. Run Cell 1 below to load the results.
3. Run Cell 2 to view formatted before/after comparisons for the first several examples.
4. Use Cell 3 as a template to manually score examples (XML adherence, example consistency).
5. Run Cell 4 to compute improvement summary statistics from your manual scores.

In [ ]:
# Cell 1: Load evaluation results
import json
from pathlib import Path

RESULTS_PATH = Path("../output/results/evaluation_results.json")

with open(RESULTS_PATH, "r") as f:
    results = json.load(f)

print(f"Loaded {len(results)} evaluation examples.")
print()
print("Sample entry:")
print(json.dumps(results[0], indent=2)[:1000])

In [ ]:
# Cell 2: Display before/after comparisons (formatted)
NUM_EXAMPLES_TO_SHOW = 10

for i, example in enumerate(results[:NUM_EXAMPLES_TO_SHOW]):
    print(f"Example {example['example_id']}")
    print("=" * 80)
    print(f"User Input: {example['user_input']}")
    print()
    print(f"Expected: {example['expected_response']}")
    print()
    print(f"[BASE MODEL]: {example['base_response']}")
    print()
    print(f"[FINE-TUNED]: {example['finetuned_response']}")
    print()
    print(f"Notes: {example['notes']}")
    print("=" * 80)
    print()

In [ ]:
# Cell 3: Manual scoring template
#
# For each example you review, fill in a scoring dict like the one below.
# Scoring scale (0-2):
#   0 = does not adhere / not consistent
#   1 = partially adheres / partially consistent
#   2 = fully adheres / fully consistent
#
# xml_adherence: does the response follow the structure/rules defined in the XML system prompt?
# example_consistency: does the response match the style/tone/content shown in the <examples> block?

manual_scores = [
    {
        "example_id": 0,
        "base_xml_adherence": None,          # 0-2
        "base_example_consistency": None,    # 0-2
        "finetuned_xml_adherence": None,     # 0-2
        "finetuned_example_consistency": None,  # 0-2
        "improvement_notes": "",
    },
    # Add one dict per example reviewed (aim for ~10-15 examples).
    # {
    #     "example_id": 1,
    #     "base_xml_adherence": 1,
    #     "base_example_consistency": 0,
    #     "finetuned_xml_adherence": 2,
    #     "finetuned_example_consistency": 2,
    #     "improvement_notes": "Fine-tuned response matches example tone much better.",
    # },
]

manual_scores

In [ ]:
# Cell 4: Summary statistics (improvement calculations)
# Run this after filling in manual_scores in Cell 3 above.

scored = [s for s in manual_scores if s["base_xml_adherence"] is not None]

if not scored:
    print("No manual scores filled in yet. Fill in Cell 3 first.")
else:
    try:
        import pandas as pd

        df = pd.DataFrame(scored)
        df["xml_adherence_improvement"] = df["finetuned_xml_adherence"] - df["base_xml_adherence"]
        df["example_consistency_improvement"] = (
            df["finetuned_example_consistency"] - df["base_example_consistency"]
        )

        avg_xml_improvement = df["xml_adherence_improvement"].mean()
        avg_consistency_improvement = df["example_consistency_improvement"].mean()
        clear_improvement_count = (
            (df["xml_adherence_improvement"] > 0) | (df["example_consistency_improvement"] > 0)
        ).sum()

        print(f"Examples scored: {len(df)}")
        print(f"Avg XML Adherence Improvement: {avg_xml_improvement:.2f}")
        print(f"Avg Example Consistency Improvement: {avg_consistency_improvement:.2f}")
        print(f"Count of examples with clear improvement: {clear_improvement_count}")

        display(df)
    except ImportError:
        # Fallback if pandas is not available
        n = len(scored)
        xml_improvements = [
            s["finetuned_xml_adherence"] - s["base_xml_adherence"] for s in scored
        ]
        consistency_improvements = [
            s["finetuned_example_consistency"] - s["base_example_consistency"] for s in scored
        ]

        avg_xml_improvement = sum(xml_improvements) / n
        avg_consistency_improvement = sum(consistency_improvements) / n
        clear_improvement_count = sum(
            1 for x, c in zip(xml_improvements, consistency_improvements) if x > 0 or c > 0
        )

        print(f"Examples scored: {n}")
        print(f"Avg XML Adherence Improvement: {avg_xml_improvement:.2f}")
        print(f"Avg Example Consistency Improvement: {avg_consistency_improvement:.2f}")
        print(f"Count of examples with clear improvement: {clear_improvement_count}")